# LangChain 기본과 Output Parser
**회의록 정리기(MeetNote)를 만들며 배우기**

### 오늘의 목표
1. LangChain의 기본 구성 요소(ChatModel, Prompt, Chain)를 이해한다
2. LLM 출력이 *문자열*이라서 생기는 문제를 확인한다
3. Output Parser로 출력을 **코드에서 쓸 수 있는 구조화 데이터**로 바꾼다

> 오늘 쓰는 예제는 처음부터 끝까지 **같은 회의 대화**입니다. 출력 형식만 점점 엄격해집니다.

## 0. 준비
```bash
uv add langchain langchain-openai pydantic
```

In [ ]:
from dotenv import load_dotenv
import os

from dotenv import load_dotenv
import os

load_dotenv(override=True)
apikey= os.getenv('OPENAI_API_KEY')
default_model = os.getenv('OPENAI_DEFAULT_MODEL')

In [ ]:
from langchain.chat_models import init_chat_model

# 모델/API 키는 환경에 이미 설정되어 있다고 가정
llm = init_chat_model(default_model)


In [ ]:
meeting_1 = """[팀 프로젝트 주간 회의 - 10/12]
민수: 이번 주 안건은 프론트엔드 일정이랑 DB 설계 확정이에요.
지연: DB는 SQLite로 가기로 했잖아요. 테이블 설계는 제가 이번 주 금요일까지 끝낼게요.
민수: 좋아요. 그럼 로그인 화면은 제가 수요일까지 만들게요.
하준: 저는 발표 자료 초안을 다음 주 월요일까지 작성하겠습니다.
지연: 그럼 배포는 Render로 하는 걸로 확정할까요?
민수: 네, Render로 확정!"""

meeting_2 = """[스터디 모임]
소영: 다음 주제는 RAG로 할까요?
태호: 좋아요. 근데 자료는 누가 준비하죠?
소영: 음... 일단 누가 하든 해야 할 것 같아요. 시간 되는 사람이?
태호: 저도 이번 주는 바빠서...
소영: 그럼 일단 RAG 하는 걸로 하고, 자료 준비는 나중에 정해요.
태호: 네 알겠어요. 장소는 어디서 하죠?
소영: 그건 다음에 얘기해요."""

meeting_3 = """[팀 프로젝트 회의]
유나: 아 점심 뭐 먹었어요? 저는 김치찌개요 ㅋㅋ
도윤: 저는 편의점... 아무튼 UI는 Streamlit으로 갈까요?
유나: 좋아요, Streamlit으로 합시다.
도윤: 아 근데 방금 찾아보니까 Gradio가 챗봇 만들기 더 편하대요. Gradio로 바꿀까요?
유나: 오 그럼 Gradio로 하죠. Streamlit은 취소!
도윤: 그럼 UI는 제가 다음 주 수요일까지 만들게요.
유나: 저는 README 작성할게요. 기한은 따로 없어요."""

samples = {"1. 깔끔한 회의": meeting_1, "2. 정보 누락 회의": meeting_2, "3. 잡담+결정 번복": meeting_3}

---
## Part 1. LangChain 기본
### 1-1. ChatModel — 모델 호출

In [ ]:
response = llm.invoke("LangChain을 한 문장으로 설명해줘")
print(type(response))
print(response.content)


> 반환값은 문자열이 아니라 **`AIMessage` 객체**입니다. 실제 텍스트는 `.content`에 들어 있어요.

### 1-2. PromptTemplate — 변수가 있는 프롬프트

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 {meeting_type} 회의록 정리 전문가다. 대화에 없는 내용은 추측하지 마라."),
    ("human", "다음 회의 내용을 정리해줘:\n\n{transcript}"),
])

# 템플릿이 변수를 채워서 메시지 리스트를 만든다 (아직 LLM 호출 전)
prompt.invoke({"meeting_type": "팀 프로젝트", "transcript": "(회의 내용)"})


### 1-3. Chain (LCEL) — `|` 로 연결
`프롬프트 | 모델` 처럼 **앞의 출력이 뒤의 입력**이 되도록 이어 붙입니다.

In [ ]:
chain = prompt | llm

result = chain.invoke({"meeting_type": "팀 프로젝트", "transcript": meeting_1})
print(result.content)


---
## Part 2. 문제 제기 — 문자열 출력의 한계
결과가 그럴듯해 보이지만, **코드로 쓰려고 하면** 막힙니다.

In [ ]:

text = result.content
print(type(text))


# 액션 아이템만 뽑아서 담당자별로 알림을 보내고 싶다면?
# text["action_items"]   # <- TypeError: 문자열이라 불가능

- 실행할 때마다 **형식이 달라짐** (표, 글머리표, 문단...)
- 담당자/기한을 **코드로 꺼낼 수 없음**
- DB 저장, UI 표시, 알림 발송 등 **후속 처리가 불가능**

👉 해결책: **Output Parser** — 모델 출력을 원하는 타입으로 변환

---
## Part 3. Output Parser
### 3-1. `StrOutputParser` — `AIMessage` → `str`
가장 단순한 파서. 체인 끝에 붙이면 `.content`를 꺼내는 수고가 사라집니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()
out = chain.invoke({"meeting_type": "팀 프로젝트", "transcript": meeting_1})
print(type(out))

### 3-2. `JsonOutputParser` — `str` → `dict`
프롬프트로 "JSON으로 답해"라고 요청하고, 파서가 `dict`로 바꿔줍니다.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 회의록 정리 전문가다. 반드시 JSON으로만 답하라. "
            "키는 summary(문자열), decisions(문자열 리스트), action_items(리스트) 이다."),
    ("human", "{transcript}"),
])

chain = json_prompt | llm | JsonOutputParser()
data = chain.invoke({"transcript": meeting_1})

print(type(data))
print(data["decisions"])

> ✅ 이제 `data["decisions"]` 로 접근할 수 있습니다.
> ⚠️ 하지만 **어떤 키가 들어올지, 타입이 맞는지는 보장되지 않습니다.**

### 3-3. `PydanticOutputParser` — 타입까지 검증
먼저 **스키마**(원하는 출력의 모양)를 Pydantic으로 정의합니다.

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

class ActionItem(BaseModel):
    task: str = Field(description="해야 할 일")
    assignee: Optional[str] = Field(None, description="담당자 이름. 대화에 언급이 없으면 null")
    due_date: Optional[str] = Field(None, description="기한. 대화에 언급이 없으면 null")

class MeetingNotes(BaseModel):
    title: str = Field(description="회의를 한 줄로 요약한 제목")
    summary: str = Field(description="3문장 이내 요약")
    decisions: list[str] = Field(description="합의된 결정 사항")
    action_items: list[ActionItem]
    open_questions: list[str] = Field(description="결론이 나지 않은 안건")

파서는 스키마를 바탕으로 **형식 지시문**을 자동 생성해 줍니다. 프롬프트에 이 지시문을 끼워 넣는 방식이에요.

In [1]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=MeetingNotes)
print(parser.get_format_instructions())

NameError: name 'MeetingNotes' is not defined

In [ ]:
pyd_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 {meeting_type} 회의록 정리 전문가다. 대화에 없는 내용은 추측하지 마라.\n{format_instructions}"),
    ("human", "{transcript}"),
]).partial(format_instructions=parser.get_format_instructions())

chain = pyd_prompt | llm | parser
notes = chain.invoke({"meeting_type": "팀 프로젝트", "transcript": meeting_1})

print(type(notes))
for item in notes.action_items:
    print(f"- {item.assignee}: {item.task} (기한: {item.due_date})")

> ✅ 반환값이 **`MeetingNotes` 객체** → 자동완성, 타입 검증, 속성 접근(`notes.title`)이 가능합니다.

### 3-4. 파싱이 실패하는 경우 (직접 확인)
LLM은 가끔 JSON 앞뒤에 **말을 덧붙입니다.** 일부러 그런 출력을 만들어 파서에 넣어 봅시다.

In [ ]:
from langchain_core.exceptions import OutputParserException

bad_output = '물론이죠! 정리해 드릴게요.\n{"title": "주간 회의", "summary": "요약"}'

try:
    parser.parse(bad_output)
except OutputParserException as e:
    print("파싱 실패:", str(e)[:200])

이런 문제를 줄이기 위해 **모델이 구조화 출력을 직접 지원하는 방식**이 나왔습니다.

### 3-5. `with_structured_output` — 권장 방식
모델의 구조화 출력(tool calling) 기능을 이용해, **파서와 형식 지시문 없이** 스키마만 넘깁니다.

In [ ]:
structured_llm = llm.with_structured_output(MeetingNotes)

simple_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 {meeting_type} 회의록 정리 전문가다. 대화에 없는 내용은 추측하지 마라."),
    ("human", "{transcript}"),
])

chain = simple_prompt | structured_llm
notes = chain.invoke({"meeting_type": "팀 프로젝트", "transcript": meeting_1})
notes

In [ ]:
print(notes.title)
print(notes.summary)
print(notes.decisions)
for item in notes.action_items:
    print(item)

### 정보가 누락된 회의에서는? — `Field(description=...)`의 효과

In [ ]:
notes2 = chain.invoke({"meeting_type": "스터디", "transcript": meeting_2})

for item in notes2.action_items:
    print(item)
print("미해결:", notes2.open_questions)

> 💡 `Field(description="... 언급이 없으면 null")` 이 **모델에게 주는 지시문** 역할을 합니다.
> 스키마 설계 = 프롬프트 설계.

---
## 정리
| 방식 | 반환 타입 | 형식 지시 | 안정성 | 언제? |
|---|---|---|---|---|
| `StrOutputParser` | `str` | 없음 | - | 그냥 텍스트면 충분할 때 |
| `JsonOutputParser` | `dict` | 프롬프트로 직접 | 낮음 | 빠른 실험 |
| `PydanticOutputParser` | Pydantic 객체 | 파서가 생성 | 중간 | 구조화 출력 미지원 모델 |
| `with_structured_output` | Pydantic 객체 | 모델 기능 활용 | 높음 | **기본 선택지** |

### 다음: 실습 과제
`MeetNote_과제_starter.ipynb` 에서 위 내용을 직접 구현해 봅니다.